# LeetCode 124 — Binary Tree Maximum Path Sum bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def maxPathSum(self, root: Optional[TreeNode]) -> int:

```

Mục tiêu của bài học:

1. Hiểu **cây nhị phân** (binary tree) từ con số 0: node, root, con trái/phải, lá, cây con; tự dựng cây từ list kiểu LeetCode và **in cây ra màn hình** để nhìn thấy nó.
2. Nắm chắc **đệ quy** (recursion) trên cây và ba kiểu duyệt **DFS**: pre-order, in-order, đặc biệt là **post-order** (hậu thứ tự — "hỏi các con trước, rồi mới quyết định ở cha").
3. Hiểu chính xác "đường đi" (path) trong đề là gì và vì sao mỗi đường đi có đúng một **đỉnh** (node cao nhất).
4. Đi qua 4 cách giải theo thứ tự tự nhiên: **brute force trên đồ thị** `O(n²)` → **thử từng đỉnh** `O(n²)` → **một lượt DFS post-order** `O(n)` (cách chính) → **bản lặp dùng stack** (không lo giới hạn đệ quy).
5. Phân biệt rõ hai con số ở mỗi node: **giá trị trả về cho cha** (chỉ một nhánh) và **giá trị dùng để cập nhật đáp án** (hai nhánh) — đây là chìa khóa của cả một họ bài "tree DP".
6. Tránh các lỗi kinh điển (khởi tạo đáp án bằng 0, quên `max(gain, 0)`, trả về cả hai nhánh, quên `nonlocal`...) và nhận dạng pattern để giải các bài anh em như Diameter of Binary Tree, Longest Univalue Path.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Một **đường đi** (path) trong cây nhị phân là một dãy các node, trong đó mỗi cặp node **kề nhau** trong dãy đều được nối với nhau bằng một cạnh. Mỗi node chỉ được xuất hiện trong dãy **nhiều nhất một lần**. Lưu ý: đường đi **không bắt buộc phải đi qua root**.
>
> **Tổng đường đi** (path sum) là tổng các giá trị của các node trên đường đi.
>
> Cho `root` của một cây nhị phân, hãy trả về **tổng đường đi lớn nhất** trong số mọi đường đi **khác rỗng**.

**Example 1**

```text
Input: root = [1,2,3]
Output: 6
Giải thích: Đường đi tối ưu là 2 -> 1 -> 3, tổng 2 + 1 + 3 = 6.
```

**Example 2**

```text
Input: root = [-10,9,20,null,null,15,7]
Output: 42
Giải thích: Đường đi tối ưu là 15 -> 20 -> 7, tổng 15 + 20 + 7 = 42.
```

**Constraints**

- Số node trong cây nằm trong khoảng `[1, 3 * 10^4]`.
- `-1000 <= Node.val <= 1000`

### Ký hiệu dùng suốt notebook

- `n` = số node của cây (theo đề `1 <= n <= 3 * 10^4`, tức là cây **không bao giờ rỗng**).
- `h` = chiều cao của cây (số node trên đường dài nhất từ root xuống lá). Cây cân bằng thì `h ≈ log₂ n`, cây "lệch" thành một đường thẳng thì `h = n`.

Hai điều cần để ý ngay từ đầu:

| Chi tiết của đề | Hệ quả |
|---|---|
| giá trị node có thể **âm** | không phải cứ đi càng dài càng tốt; đôi khi nên "cắt bỏ" một nhánh |
| đường đi **khác rỗng** | luôn phải chọn ít nhất 1 node, nên cây `[-3]` có đáp án `-3`, **không phải** `0` |
| không bắt buộc qua root | Example 2: root `-10` không nằm trong đường đi tốt nhất |

### 1.1 "Đường đi" trông như thế nào?

Vẽ cây của Example 2 (LeetCode in cây theo từng tầng, `null` là "không có con ở vị trí này"):

```text
        -10
       /    \
      9      20
            /  \
          15    7
```

Một đường đi giống như **một sợi dây** đặt dọc theo các cạnh: nó có hai đầu, không rẽ nhánh, không đi qua node nào hai lần. Nó có thể đi **lên** rồi **xuống**.

| Dãy node | Hợp lệ? | Tổng | Lý do |
|---|:---:|---:|---|
| `15 -> 20 -> 7` | ✅ | 42 | lên từ 15 tới 20, rồi xuống 7 |
| `9 -> -10 -> 20 -> 15` | ✅ | 34 | lên từ 9 tới -10, rồi xuống 20, xuống 15 |
| `20` | ✅ | 20 | một node đơn lẻ cũng là đường đi |
| `9 -> 20` | ❌ | — | 9 và 20 **không có cạnh nối trực tiếp** |
| `15 -> 20 -> 7` **và thêm** `-10` | ❌ | — | node 20 sẽ phải nối với 3 node (15, 7, -10) → hình chữ Y, không còn là "sợi dây" |
| `15 -> 20 -> 15` | ❌ | — | node 15 xuất hiện 2 lần |

Dòng thứ 5 là điểm mấu chốt của cả bài: **một đường đi không thể rẽ nhánh**. Ta sẽ gặp lại ý này khi phân biệt "giá trị trả về" và "giá trị cập nhật".

## 2. Nền tảng: Cây nhị phân từ căn bản

### 2.1 Từ vựng

**Cây nhị phân** (binary tree) là cấu trúc gồm các **node** (nút). Mỗi node có:

- `val`: giá trị lưu ở node.
- `left`: con **trái** (một node khác) hoặc `None` nếu không có.
- `right`: con **phải** hoặc `None`.

Một số từ cần nhớ (dùng cây Example 2 ở trên):

| Thuật ngữ | Nghĩa | Ví dụ trên cây Example 2 |
|---|---|---|
| **root** (gốc) | node trên cùng, không có cha | `-10` |
| **child** (con) / **parent** (cha) | node ngay dưới / ngay trên | `20` là con phải của `-10`; `-10` là cha của `20` |
| **leaf** (lá) | node không có con nào | `9`, `15`, `7` |
| **subtree** (cây con) | một node cùng toàn bộ con cháu của nó | cây con gốc `20` gồm `{20, 15, 7}` |
| **edge** (cạnh) | đoạn nối cha với con | `-10 — 20` |
| **depth** (độ sâu) | số cạnh từ root tới node | depth(`-10`)=0, depth(`20`)=1, depth(`15`)=2 |

Khác với Python `list` (truy cập bằng chỉ số `a[i]`), ở cây ta chỉ cầm **node root** và đi xuống bằng `.left` / `.right`.

LeetCode định nghĩa node bằng class `TreeNode` (cho sẵn, được comment ở đầu đề). Ta định nghĩa lại **y hệt** để chạy trong notebook.

`Optional[TreeNode]` trong signature (từ module `typing`) nghĩa là "hoặc là một `TreeNode`, hoặc là `None`" — chỉ là chú thích kiểu cho người đọc, Python không bắt buộc.

In [1]:
from typing import List, Optional
from collections import deque
import random
import sys
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


# Dựng tay cây Example 1:   1
#                          / \
#                         2   3
root1 = TreeNode(1, TreeNode(2), TreeNode(3))

print(root1.val)          # 1
print(root1.left.val)     # 2
print(root1.right.val)    # 3
print(root1.left.left)    # None -> node 2 không có con trái (2 là lá)

1
2
3
None


### 2.2 LeetCode biểu diễn cây bằng list theo tầng (level-order)

Trên LeetCode, `[-10,9,20,null,null,15,7]` được đọc **từng tầng, từ trái sang phải**:

```text
tầng 0:  -10
tầng 1:  9, 20                  (con trái, con phải của -10)
tầng 2:  null, null, 15, 7      (con của 9 là null, null; con của 20 là 15, 7)
```

Quy tắc: lấy lần lượt từng node **thật** (không phải `null`) theo thứ tự tầng, mỗi node "tiêu thụ" 2 phần tử kế tiếp của list làm con trái và con phải. `null` không có con, nên không tiêu thụ thêm phần tử nào. Phần `null` ở cuối list thường bị lược bỏ.

Để làm đúng thứ tự "node nào đến trước thì được gán con trước", ta dùng **hàng đợi** (queue): vào trước, ra trước (FIFO — First In, First Out). Python có sẵn `collections.deque`:

- `q.append(x)`: thêm `x` vào **cuối** hàng.
- `q.popleft()`: lấy ra phần tử ở **đầu** hàng (nhanh `O(1)`, còn `list.pop(0)` thì chậm `O(n)`).

In [2]:
q = deque()
q.append("A")
q.append("B")
q.append("C")
print("hàng đợi:", list(q))
print("ra trước:", q.popleft())   # A vào trước nên ra trước
print("ra tiếp :", q.popleft())   # B
print("còn lại :", list(q))       # ['C']

hàng đợi: ['A', 'B', 'C']
ra trước: A
ra tiếp : B
còn lại : ['C']


### 2.3 Helper: `build_tree` (list → cây) và `tree_to_list` (cây → list)

- `build_tree(values)`: tạo root từ `values[0]`, đưa vào queue; mỗi lần `popleft` một node thì đọc 2 phần tử tiếp theo làm con trái/phải (bỏ qua `None`), con thật được đưa vào cuối queue để lát nữa tới lượt nó nhận con.
- `tree_to_list(root)`: làm ngược lại để kiểm tra — duyệt theo tầng, ghi `None` cho chỗ trống, rồi cắt các `None` thừa ở cuối.

Nếu `build_tree` rồi `tree_to_list` ra lại đúng list ban đầu, ta tin được helper.

In [3]:
def build_tree(values):
    # Dựng cây từ list level-order kiểu LeetCode (None = không có node).
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:      # con trái
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:      # con phải
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    # Cây -> list level-order (để kiểm tra build_tree).
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:      # bỏ None thừa ở cuối
        out.pop()
    return out


ex2 = [-10, 9, 20, None, None, 15, 7]
root2 = build_tree(ex2)
print(tree_to_list(root2))
print(root2.right.left.val)     # đi: -10 -> phải (20) -> trái (15)
assert tree_to_list(build_tree(ex2)) == ex2
assert tree_to_list(build_tree([1, 2, 3])) == [1, 2, 3]
assert tree_to_list(build_tree([1, None, 2, None, 3])) == [1, None, 2, None, 3]
print("build_tree / tree_to_list OK")

[-10, 9, 20, None, None, 15, 7]
15
build_tree / tree_to_list OK


### 2.4 Helper: in cây dạng ASCII để **nhìn thấy** cây

Đọc list `[-10,9,20,null,null,15,7]` rất khó hình dung. Hàm `print_tree` in cây theo kiểu thư mục: mỗi node một dòng, `L:` là con trái, `R:` là con phải, `·` nghĩa là "không có con ở phía này" (chỉ in khi phía kia có con, để thấy rõ trái/phải).

Hàm này tự gọi lại chính nó cho từng con — đó chính là **đệ quy**, ta sẽ học kỹ ngay ở mục 3. Bây giờ chỉ cần dùng.

In [4]:
def print_tree(root):
    if root is None:
        print("(cây rỗng)")
        return
    print(root.val)

    def walk(node, prefix):
        if node.left is None and node.right is None:
            return
        for tag, child, last in (("L", node.left, False), ("R", node.right, True)):
            branch = "└── " if last else "├── "
            if child is None:
                print(prefix + branch + tag + ": ·")
            else:
                print(prefix + branch + f"{tag}: {child.val}")
                walk(child, prefix + ("    " if last else "│   "))

    walk(root, "")


print("Example 1:")
print_tree(build_tree([1, 2, 3]))
print("\nExample 2:")
print_tree(root2)
print("\nCây lệch [1,null,2,null,3]:")
print_tree(build_tree([1, None, 2, None, 3]))

Example 1:
1
├── L: 2
└── R: 3

Example 2:
-10
├── L: 9
└── R: 20
    ├── L: 15
    └── R: 7

Cây lệch [1,null,2,null,3]:
1
├── L: ·
└── R: 2
    ├── L: ·
    └── R: 3


## 3. Nền tảng: Đệ quy (recursion) trên cây

### 3.1 Ý tưởng

**Đệ quy** là khi một hàm **gọi lại chính nó** trên một bài toán **nhỏ hơn**. Cây rất hợp với đệ quy vì: *cây = root + cây con trái + cây con phải*, và hai cây con cũng là cây.

Một hàm đệ quy luôn có 2 phần:

1. **Base case** (trường hợp cơ sở): bài toán nhỏ tới mức trả lời ngay. Với cây, gần như luôn là `if node is None: return ...`.
2. **Recursive case**: gọi hàm cho `node.left`, `node.right`, rồi **kết hợp** hai kết quả với `node.val`.

Ví dụ: tính tổng mọi giá trị trong cây.

```text
tree_sum(None) = 0                                        # base case
tree_sum(node) = node.val + tree_sum(left) + tree_sum(right)
```

Với Example 2: `tree_sum(20) = 20 + 15 + 7 = 42`, `tree_sum(9) = 9`, `tree_sum(-10) = -10 + 9 + 42 = 41`.

In [5]:
def count_nodes(node):
    if node is None:            # base case: cây rỗng có 0 node
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)


def tree_sum(node):
    if node is None:            # base case: tổng cây rỗng = 0
        return 0
    return node.val + tree_sum(node.left) + tree_sum(node.right)


print("số node Example 2:", count_nodes(root2))   # 5
print("tổng   Example 2:", tree_sum(root2))       # 41
print("tổng cây con 20 :", tree_sum(root2.right)) # 42

số node Example 2: 5
tổng   Example 2: 41
tổng cây con 20 : 42


### 3.2 Call stack: máy tính "chờ" các lời gọi con như thế nào?

Khi `tree_sum(-10)` gọi `tree_sum(9)`, nó **tạm dừng** và chờ. Các lời gọi đang chờ được xếp chồng lên nhau trong **call stack** (ngăn xếp lời gọi). Lời gọi nào vào sau thì xong trước.

Cell dưới in ra mỗi lần **vào** và **ra** khỏi một lời gọi, thụt lề theo độ sâu, để thấy rõ: một node chỉ "ra" (trả kết quả) **sau khi cả hai con đã ra**.

In [6]:
def tree_sum_traced(node, depth=0):
    pad = "    " * depth
    if node is None:
        return 0
    print(f"{pad}vào {node.val}")
    s = node.val + tree_sum_traced(node.left, depth + 1) + tree_sum_traced(node.right, depth + 1)
    print(f"{pad}ra  {node.val} -> trả {s}")
    return s


tree_sum_traced(root2)

vào -10
    vào 9
    ra  9 -> trả 9
    vào 20
        vào 15
        ra  15 -> trả 15
        vào 7
        ra  7 -> trả 7
    ra  20 -> trả 42
ra  -10 -> trả 41


41

### 3.3 Ba kiểu duyệt DFS: pre-order, in-order, post-order

**DFS** (Depth-First Search — duyệt theo chiều sâu) nghĩa là đi xuống sâu hết một nhánh rồi mới quay lại nhánh khác. Đệ quy ở trên chính là DFS.

Cùng một khung đệ quy, chỉ cần đổi **vị trí của dòng "xử lý node"** là ra ba thứ tự khác nhau:

| Tên | Thứ tự | Khi nào dùng |
|---|---|---|
| **pre-order** (tiền thứ tự) | node → trái → phải | cha cần truyền thông tin **xuống** con |
| **in-order** (trung thứ tự) | trái → node → phải | cây nhị phân tìm kiếm (BST) cho ra dãy tăng dần |
| **post-order** (hậu thứ tự) | trái → phải → node | cha cần **kết quả của các con** rồi mới tính được |

Với Example 2, dự đoán trước: pre-order `[-10, 9, 20, 15, 7]`, in-order `[9, -10, 15, 20, 7]`, post-order `[9, 15, 7, 20, -10]`.

In [7]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # xử lý node TRƯỚC
    preorder(node.left, out)
    preorder(node.right, out)


def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)        # xử lý node GIỮA
    inorder(node.right, out)


def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)        # xử lý node SAU CÙNG


for name, fn in [("pre-order ", preorder), ("in-order  ", inorder), ("post-order", postorder)]:
    out = []
    fn(root2, out)
    print(name, out)

pre-order  [-10, 9, 20, 15, 7]
in-order   [9, -10, 15, 20, 7]
post-order [9, 15, 7, 20, -10]


### 3.4 Post-order = "hỏi các con trước, rồi mới quyết định ở cha"

Bài 124 là bài **post-order**: muốn biết "đi xuống từ node này được tổng lớn nhất bao nhiêu", ta phải biết câu trả lời của **con trái** và **con phải** trước.

Ví dụ khởi động: **chiều cao** của cây (số node trên đường dài nhất từ node xuống lá).

```text
height(None) = 0
height(node) = 1 + max(height(left), height(right))
```

Để ý chữ `max`: cha **chọn một trong hai** nhánh con (nhánh cao hơn). Đây là dạng "kết hợp kết quả của con" mà bài chính sẽ dùng.

In [8]:
def height(node):
    if node is None:
        return 0
    left_h = height(node.left)       # hỏi con trái
    right_h = height(node.right)     # hỏi con phải
    return 1 + max(left_h, right_h)  # rồi mới quyết định ở cha


print("height Example 1:", height(build_tree([1, 2, 3])))              # 2
print("height Example 2:", height(root2))                              # 3
print("height cây lệch :", height(build_tree([1, None, 2, None, 3])))  # 3

height Example 1: 2
height Example 2: 3
height cây lệch : 3


### 3.5 Khởi động gần với bài chính: tổng lớn nhất từ root **xuống** một lá

Đổi `1 + max(...)` thành `node.val + max(...)` là ta có "đường đi xuống từ root tới một lá có tổng lớn nhất":

```text
best_root_to_leaf(lá)  = lá.val
best_root_to_leaf(node) = node.val + max(của các con đang tồn tại)
```

Với Example 2: nhánh trái `-10 + 9 = -1`; nhánh phải `-10 + 20 + 15 = 25` (tốt hơn `-10 + 20 + 7 = 17`). Kết quả `25`.

Chú ý: node chỉ có một con thì **không được** lấy `max` với nhánh `None`, vì đường đi phải kết thúc ở lá. (Bài 124 sẽ thoải mái hơn: được dừng ở bất kỳ node nào.)

In [9]:
def best_root_to_leaf(node):
    if node.left is None and node.right is None:     # lá
        return node.val
    candidates = []
    if node.left:
        candidates.append(best_root_to_leaf(node.left))
    if node.right:
        candidates.append(best_root_to_leaf(node.right))
    return node.val + max(candidates)


print(best_root_to_leaf(root2))                       # 25
print(best_root_to_leaf(build_tree([1, 2, 3])))       # 4  (1 -> 3)

25
4


### 3.6 Cập nhật một biến "bên ngoài" từ trong hàm đệ quy: `nonlocal`

Lời giải chính sẽ cần một biến `best` (đáp án tốt nhất tới giờ) nằm **ngoài** hàm đệ quy, và hàm đệ quy **ghi đè** nó ở mỗi node.

Trong Python, nếu một hàm lồng bên trong (nested function) muốn **gán lại** biến của hàm bao ngoài, phải khai báo `nonlocal tên_biến`. Nếu quên, Python coi `best` là biến cục bộ mới → lỗi `UnboundLocalError`.

Ví dụ mini: tìm giá trị node lớn nhất bằng một biến `biggest` bên ngoài.

In [10]:
def max_value(root):
    biggest = float("-inf")       # -vô cực: nhỏ hơn mọi số

    def visit(node):
        nonlocal biggest          # cho phép gán lại biến của hàm bao ngoài
        if node is None:
            return
        biggest = max(biggest, node.val)
        visit(node.left)
        visit(node.right)

    visit(root)
    return biggest


def max_value_forgot_nonlocal(root):
    biggest = float("-inf")

    def visit(node):
        if node is None:
            return
        biggest = max(biggest, node.val)   # SAI: thiếu nonlocal
        visit(node.left)
        visit(node.right)

    visit(root)
    return biggest


print("max_value:", max_value(root2))       # 20
try:
    max_value_forgot_nonlocal(root2)
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

max_value: 20
UnboundLocalError: cannot access local variable 'biggest' where it is not associated with a value


## 4. Cách 1 — Brute force: biến cây thành đồ thị, thử mọi đường đi

### 4.1 Ý tưởng tự nhiên nhất

Người mới thường nghĩ: "Một đường đi có **hai đầu** `u` và `v` (có thể `u = v`). Vậy cứ thử mọi cặp đầu mút, tính tổng đường đi giữa chúng, lấy lớn nhất."

Trong cây, giữa hai node bất kỳ có **đúng một** đường đi không lặp node (vì cây không có chu trình). Nên "mọi đường đi" = "mọi cặp `(u, v)`".

Khó khăn: trong `TreeNode`, ta chỉ đi được **xuống** (`left`, `right`), không đi **lên** cha được. Mà đường đi `9 -> -10 -> 20` lại cần đi lên từ `9` tới `-10`.

Cách gỡ: chuyển cây thành **đồ thị vô hướng** (undirected graph) — mỗi node biết hết **hàng xóm** của nó (cha và các con). Ta lưu bằng `dict`: key là node, value là list các node kề.

Lưu ý: key là **chính đối tượng node**, không phải giá trị `val`, vì hai node khác nhau có thể có cùng giá trị.

In [11]:
def tree_to_graph(root):
    graph = {}

    def add(node, parent):
        if node is None:
            return
        graph[node] = []
        if parent is not None:          # cạnh hai chiều cha <-> con
            graph[node].append(parent)
            graph[parent].append(node)
        add(node.left, node)
        add(node.right, node)

    add(root, None)
    return graph


g2 = tree_to_graph(root2)
for node, neighbors in g2.items():
    print(f"{node.val:>4} kề với {[nb.val for nb in neighbors]}")

 -10 kề với [9, 20]
   9 kề với [-10]
  20 kề với [-10, 15, 7]
  15 kề với [20]
   7 kề với [20]


### 4.2 DFS từ mỗi đỉnh xuất phát

Cố định một đầu mút `start`. Từ `start` đi DFS ra khắp đồ thị, mang theo **tổng tích lũy** `total`. Mỗi node `v` ta chạm tới chính là đầu mút thứ hai của đường `start → v`, nên cập nhật `best = max(best, total)` ở **mọi** node chạm tới.

Để không quay ngược lại, chỉ cần nhớ node vừa đi qua (`parent`) và không đi về nó. Trong cây như vậy là đủ (không cần `visited` set), vì cây không có chu trình nên không có cách nào khác để quay lại một node cũ.

Ví dụ `start = 9`: đi `9` (total 9) → `-10` (total -1) → `20` (total 19) → `15` (total 34) và `7` (total 26). Tốt nhất từ `9` là `34`.

Làm vậy cho cả `n` điểm xuất phát.

In [12]:
def max_path_brute_graph(root):
    graph = tree_to_graph(root)
    best = float("-inf")

    def explore(node, parent, total):
        nonlocal best
        best = max(best, total)                # đường start -> node có tổng total
        for nb in graph[node]:
            if nb is not parent:               # không đi ngược lại
                explore(nb, node, total + nb.val)

    for start in graph:                        # thử mọi điểm xuất phát
        explore(start, None, start.val)
    return best


print(max_path_brute_graph(build_tree([1, 2, 3])))                   # 6
print(max_path_brute_graph(root2))                                   # 42
print(max_path_brute_graph(build_tree([-3])))                        # -3
print(max_path_brute_graph(build_tree([2, -1])))                     # 2

6
42
-3
2


### 4.3 Brute force vướng ở đâu?

- Từ mỗi điểm xuất phát, DFS đi qua **toàn bộ** `n` node → tổng cộng `n · n = n²` lần gọi `explore`.
- Mỗi đường đi `u — v` bị tính **hai lần** (xuất phát từ `u` và từ `v`), và các đoạn chung bị cộng lại rất nhiều lần.
- Với `n = 3 · 10^4`: `n² = 9 · 10^8` bước — quá chậm với Python (TLE — Time Limit Exceeded).

**Time** `O(n²)`, **Space** `O(n)` (đồ thị + độ sâu đệ quy tối đa `n`).

Cell dưới đếm số lần gọi `explore` trên cây lệch (một đường thẳng) `n` node: khi `n` gấp đôi, số bước **gấp bốn** — dấu hiệu của `O(n²)`.

In [13]:
def make_chain(n, val=1):
    # Cây lệch: 1 -> 1 -> ... (toàn con phải), dựng bằng vòng lặp
    root = TreeNode(val)
    cur = root
    for _ in range(n - 1):
        cur.right = TreeNode(val)
        cur = cur.right
    return root


def count_brute_steps(root):
    graph = tree_to_graph(root)
    steps = 0

    def explore(node, parent):
        nonlocal steps
        steps += 1
        for nb in graph[node]:
            if nb is not parent:
                explore(nb, node)

    for start in graph:
        explore(start, None)
    return steps


for n in [100, 200, 400]:
    print(f"n = {n:>3}: số lần explore = {count_brute_steps(make_chain(n)):>6}  (n² = {n * n})")

n = 100: số lần explore =  10000  (n² = 10000)
n = 200: số lần explore =  40000  (n² = 40000)
n = 400: số lần explore = 160000  (n² = 160000)


## 5. Cách 2 — Mỗi đường đi có một "đỉnh": thử từng node làm đỉnh

### 5.1 Quan sát then chốt: đỉnh của đường đi

Nhìn lại cây Example 2 với độ sâu từng node:

```text
        -10          depth 0
       /    \
      9      20      depth 1
            /  \
          15    7    depth 2
```

Mỗi đường đi có **đúng một** node **cao nhất** (depth nhỏ nhất) — gọi là **đỉnh** (apex, hay "điểm gập"). Vì sao duy nhất? Nếu có hai node cùng cao nhất trên đường đi thì để đi từ node này sang node kia phải đi qua cha chung của chúng, mà cha chung còn cao hơn — mâu thuẫn.

Từ đỉnh, đường đi chỉ có thể **đi xuống**, tối đa theo **hai hướng**: một đoạn xuống cây con trái, một đoạn xuống cây con phải (mỗi đoạn có thể rỗng).

| Đường đi | Đỉnh | Đoạn xuống trái | Đoạn xuống phải |
|---|---|---|---|
| `15 -> 20 -> 7` | `20` | `15` | `7` |
| `9 -> -10 -> 20 -> 15` | `-10` | `9` | `20 -> 15` |
| `20 -> 15` | `20` | `15` | (rỗng) |
| `9` | `9` | (rỗng) | (rỗng) |

Như vậy: **đáp án = max theo mọi node `x`, của "đường đi tốt nhất có đỉnh là `x`"**. Bài toán khó (mọi đường đi) được chẻ thành `n` bài nhỏ (mỗi đỉnh một bài).

In [14]:
def node_depths(root):
    depth = {}

    def go(node, d):
        if node is None:
            return
        depth[node] = d
        go(node.left, d + 1)
        go(node.right, d + 1)

    go(root, 0)
    return depth


n_m10, n_9, n_20 = root2, root2.left, root2.right
n_15, n_7 = n_20.left, n_20.right
depth2 = node_depths(root2)

for path in ([n_15, n_20, n_7], [n_9, n_m10, n_20, n_15], [n_20, n_15]):
    apex = min(path, key=lambda node: depth2[node])        # node có depth nhỏ nhất
    print([node.val for node in path], "-> đỉnh =", apex.val)

[15, 20, 7] -> đỉnh = 20
[9, -10, 20, 15] -> đỉnh = -10
[20, 15] -> đỉnh = 20


### 5.2 "Gain": đi xuống từ một node được nhiều nhất bao nhiêu?

Định nghĩa:

> `gain(x)` = tổng lớn nhất của một đường đi **bắt đầu tại `x` và chỉ đi xuống** (dừng ở đâu cũng được, ít nhất gồm chính `x`).

Từ `x`, ta có 3 lựa chọn: dừng luôn, đi tiếp xuống trái, hoặc đi tiếp xuống phải (không được cả hai — đây là đường đi *bắt đầu* tại `x`, đi một chiều xuống). Vậy:

```text
gain(None) = 0
gain(x)    = x.val + max(0, gain(x.left), gain(x.right))
```

Số `0` trong `max` chính là lựa chọn "**dừng ở `x`**, không đi xuống nữa". Ta chọn nó khi cả hai nhánh con đều cho tổng âm — đi xuống chỉ làm tổng nhỏ đi.

Tính tay trên Example 2 (từ dưới lên — post-order!):

| Node | gain(trái) | gain(phải) | gain(node) |
|---|---:|---:|---|
| `9` | 0 | 0 | `9 + max(0, 0, 0) = 9` |
| `15` | 0 | 0 | `15` |
| `7` | 0 | 0 | `7` |
| `20` | 15 | 7 | `20 + max(0, 15, 7) = 35` |
| `-10` | 9 | 35 | `-10 + max(0, 9, 35) = 25` |

Và ví dụ nhánh âm: cây `[2, -1]` → `gain(-1) = -1`, `gain(2) = 2 + max(0, -1, 0) = 2` (không đi xuống `-1`).

In [15]:
def max_gain(node):
    if node is None:
        return 0
    return node.val + max(0, max_gain(node.left), max_gain(node.right))


for node in (n_9, n_15, n_7, n_20, n_m10):
    print(f"gain({node.val:>3}) = {max_gain(node)}")

r = build_tree([2, -1])
print("cây [2,-1]: gain(-1) =", max_gain(r.left), ", gain(2) =", max_gain(r))

gain(  9) = 9
gain( 15) = 15
gain(  7) = 7
gain( 20) = 35
gain(-10) = 25
cây [2,-1]: gain(-1) = -1 , gain(2) = 2


### 5.3 Đường đi tốt nhất có đỉnh là `x`

Đường đi có đỉnh `x` = (đoạn xuống trái, có thể rỗng) + `x` + (đoạn xuống phải, có thể rỗng). Hai đoạn nằm ở **hai cây con khác nhau**, nên chọn độc lập, mỗi bên lấy tốt nhất:

```text
through(x) = x.val + max(0, gain(x.left)) + max(0, gain(x.right))
```

Bên nào âm thì bỏ (thay bằng 0). **Bản thân `x.val` thì luôn được cộng** — đỉnh bắt buộc thuộc đường đi, kể cả khi `x.val` âm.

| Đỉnh `x` | max(0, gain trái) | max(0, gain phải) | through(x) |
|---|---:|---:|---:|
| `9` | 0 | 0 | 9 |
| `15` | 0 | 0 | 15 |
| `7` | 0 | 0 | 7 |
| `20` | 15 | 7 | **42** |
| `-10` | 9 | 35 | 34 |

Lớn nhất là `42` tại đỉnh `20` — khớp Example 2.

Cách 2: duyệt mọi node, với mỗi node gọi `max_gain` cho hai con để tính `through`, lấy max.

In [16]:
def max_path_apex_naive(root):
    best = float("-inf")

    def visit(node):
        nonlocal best
        if node is None:
            return
        left = max(0, max_gain(node.left))       # tính lại từ đầu!
        right = max(0, max_gain(node.right))     # tính lại từ đầu!
        best = max(best, node.val + left + right)
        visit(node.left)
        visit(node.right)

    visit(root)
    return best


print(max_path_apex_naive(build_tree([1, 2, 3])))    # 6
print(max_path_apex_naive(root2))                    # 42
print(max_path_apex_naive(build_tree([-3])))         # -3
print(max_path_apex_naive(build_tree([2, -1])))      # 2

6
42
-3
2


### 5.4 Cách 2 vướng ở đâu?

Ý tưởng "đỉnh" đã đúng, nhưng `max_gain` bị **tính lại** rất nhiều lần:

- Khi thử đỉnh `-10`, ta gọi `max_gain(20)` — hàm này đi hết cây con `{20, 15, 7}`.
- Khi thử đỉnh `20`, ta lại gọi `max_gain(15)` và `max_gain(7)` — tính lại đúng những thứ vừa tính.

Mỗi node `x` tốn thời gian bằng kích thước cây con của nó. Tổng = tổng kích thước mọi cây con:

- Cây cân bằng: khoảng `n log n`.
- Cây lệch (đường thẳng): `n + (n-1) + ... + 1 = n(n+1)/2` → **`O(n²)`** trong trường hợp xấu nhất.

Đếm số lần gọi `max_gain` (tính cả các lời gọi với `None`) để thấy rõ: khi `n` gấp đôi, số lần gọi gấp khoảng bốn — đúng dáng `O(n²)`.

In [17]:
def count_apex_naive_calls(root):
    calls = 0

    def gain(node):
        nonlocal calls
        calls += 1
        if node is None:
            return 0
        return node.val + max(0, gain(node.left), gain(node.right))

    def visit(node):
        if node is None:
            return
        gain(node.left)
        gain(node.right)
        visit(node.left)
        visit(node.right)

    visit(root)
    return calls


for n in [100, 200, 400]:
    print(f"cây lệch n = {n:>3}: số lần gọi gain = {count_apex_naive_calls(make_chain(n)):>6}")

cây lệch n = 100: số lần gọi gain =  10100
cây lệch n = 200: số lần gọi gain =  40200
cây lệch n = 400: số lần gọi gain = 160400


## 6. Cách 3 (cách chính) — Một lượt DFS post-order, `O(n)`

### 6.1 Mấu chốt cải tiến

Nhìn lại công thức:

```text
gain(x)    = x.val + max(0, gain(left), gain(right))
through(x) = x.val + max(0, gain(left)) + max(0, gain(right))
```

Cả hai **chỉ cần `gain(left)` và `gain(right)`**. Mà khi DFS post-order tính `gain(x)`, nó **đã có sẵn** hai số này trong tay (vừa hỏi các con xong).

Vậy thay vì tính `gain` một lần rồi lại tính lại cho từng đỉnh, ta làm **hai việc trong cùng một lời gọi**:

1. Tính `through(x)` và cập nhật đáp án toàn cục `best`.
2. Trả `gain(x)` lên cho cha dùng tiếp.

Mỗi node chỉ được thăm **đúng một lần** → `O(n)`.

### 6.2 Hai con số ở mỗi node: "trả về" khác "cập nhật"

Đây là phần dễ nhầm nhất, cần nhớ thật kỹ. Đặt `L = max(gain(trái), 0)`, `R = max(gain(phải), 0)`:

| | Công thức | Số nhánh | Ý nghĩa | Đi đâu? |
|---|---|:---:|---|---|
| **Giá trị cập nhật** | `x.val + L + R` | **2** | đường đi **gập tại `x`** (`x` là đỉnh) | `best = max(best, ...)` |
| **Giá trị trả về** | `x.val + max(L, R)` | **1** | đường đi **bắt đầu tại `x`, đi xuống**, còn có thể nối dài lên cha | `return ...` |

**Tại sao giá trị trả về chỉ được một nhánh?** Cha sẽ nối thêm chính nó vào đầu đoạn này: `cha → x → ...`. Nếu đoạn của `x` đã dùng cả trái lẫn phải, thì `x` nối với 3 node (cha, con trái, con phải) → hình chữ **Y**, không còn là đường đi (xem dòng ❌ ở mục 1.1).

**Tại sao giá trị cập nhật được hai nhánh?** Vì nó **không** được gửi lên cha. Đường gập tại `x` là một đường đi hoàn chỉnh, ta chỉ ghi nhận nó làm "ứng viên đáp án" rồi thôi.

Ví dụ ở node `20` (Example 2): `L = 15`, `R = 7`.

- Cập nhật: `20 + 15 + 7 = 42` → đường `15 -> 20 -> 7`.
- Trả về cho `-10`: `20 + max(15, 7) = 35` → đoạn `20 -> 15`, để `-10` có thể nối thành `... -10 -> 20 -> 15`.

In [18]:
L, R, val = 15, 7, 20
print("cập nhật best với (2 nhánh):", val + L + R)        # 42: 15 -> 20 -> 7
print("trả về cho cha   (1 nhánh):", val + max(L, R))     # 35: 20 -> 15

cập nhật best với (2 nhánh): 42
trả về cho cha   (1 nhánh): 35


### 6.3 Ý nghĩa của `max(gain, 0)` và vì sao `best` khởi tạo là `-inf`

**`max(gain(con), 0)`** = "nếu đi xuống nhánh con này làm tổng **giảm** thì **không đi**". Đường đi được phép dừng ở bất kỳ node nào, nên một nhánh có gain âm chỉ có hại.

Ví dụ cây `[2, -1]`: `gain(-1) = -1 < 0` → ở node `2`, `L = max(-1, 0) = 0`. Đường tốt nhất là chỉ `[2]`, tổng `2` (chứ không phải `2 + (-1) = 1`).

Chú ý ta chỉ "cắt" **nhánh con**, không cắt `x.val`: đỉnh luôn nằm trong đường đi của chính nó.

**`best = float("-inf")`** (âm vô cực), **không phải `0`**: đề yêu cầu đường đi khác rỗng. Nếu mọi node đều âm, ví dụ `[-3]` hoặc `[-2, -1]`, đáp án là số âm (`-3`, `-1`). Khởi tạo `best = 0` sẽ trả sai `0` — một "đường đi rỗng" không tồn tại. (Cũng có thể khởi tạo `best = root.val`, vì cây có ít nhất 1 node.)

`gain(None) = 0` là an toàn: con không tồn tại tương đương "không đi xuống phía đó".

In [19]:
def max_path_sum_dfs(root):
    best = float("-inf")

    def gain(node):
        nonlocal best
        if node is None:
            return 0
        left_gain = max(gain(node.left), 0)       # nhánh âm -> bỏ (coi như 0)
        right_gain = max(gain(node.right), 0)
        path_through = node.val + left_gain + right_gain   # 2 nhánh: cập nhật đáp án
        best = max(best, path_through)
        return node.val + max(left_gain, right_gain)       # 1 nhánh: trả cho cha

    gain(root)
    return best

In [20]:
tests = [
    ([1, 2, 3], 6),                        # Example 1
    ([-10, 9, 20, None, None, 15, 7], 42), # Example 2
    ([-3], -3),                            # một node âm
    ([-2, -1], -1),                        # toàn âm: chọn node lớn nhất
    ([2, -1], 2),                          # cắt nhánh âm
    ([1, -2, 3], 4),                       # 1 -> 3
]
for values, expected in tests:
    got = max_path_sum_dfs(build_tree(values))
    print(f"{str(values):<34} -> {got:>3}   (mong đợi {expected})")
    assert got == expected

[1, 2, 3]                          ->   6   (mong đợi 6)
[-10, 9, 20, None, None, 15, 7]    ->  42   (mong đợi 42)
[-3]                               ->  -3   (mong đợi -3)
[-2, -1]                           ->  -1   (mong đợi -1)
[2, -1]                            ->   2   (mong đợi 2)
[1, -2, 3]                         ->   4   (mong đợi 4)


## 7. Trace từng bước trên `[-10,9,20,null,null,15,7]`

Thứ tự xử lý là **post-order**: `9, 15, 7, 20, -10` (một node chỉ tính xong sau khi hai con đã xong). Lời gọi `gain(None)` trả 0 ngay, không ghi vào bảng.

Các cột: `L`, `R` là gain của con **sau khi** đã `max(..., 0)`; `through` = `val + L + R` (giá trị cập nhật); `best` là giá trị **sau** khi cập nhật; `return` = `val + max(L, R)` (giá trị trả về cho cha).

| Bước | Node | L | R | through = val+L+R | best sau | return = val+max(L,R) |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | `9` | 0 | 0 | 9 | 9 | 9 |
| 2 | `15` | 0 | 0 | 15 | 15 | 15 |
| 3 | `7` | 0 | 0 | 7 | 15 | 7 |
| 4 | `20` | 15 | 7 | **42** | **42** | 35 |
| 5 | `-10` | 9 | 35 | 34 | 42 | 25 |

Nhận xét:

- Bước 4: `20` cập nhật `best = 42` nhưng chỉ trả `35` lên cha — cha **không được** dùng cả `15` lẫn `7`.
- Bước 5: đường gập tại root `-10` là `9 -> -10 -> 20 -> 15 = 34 < 42`, nên `best` giữ `42`. Đáp án không đi qua root.
- Giá trị trả về cuối cùng của root (`25`) **bị bỏ đi** — không phải đáp án! Đáp án là `best`.

In [21]:
def trace_max_path(root):
    best = float("-inf")
    step = 0

    def gain(node, depth):
        nonlocal best, step
        if node is None:
            return 0
        pad = "  " * depth
        print(f"{pad}vào {node.val}")
        left_gain = max(gain(node.left, depth + 1), 0)
        right_gain = max(gain(node.right, depth + 1), 0)
        through = node.val + left_gain + right_gain
        best = max(best, through)
        ret = node.val + max(left_gain, right_gain)
        step += 1
        print(f"{pad}[bước {step}] node={node.val}: L={left_gain}, R={right_gain}, "
              f"through={through}, best={best}, return={ret}")
        return ret

    gain(root, 0)
    print("Đáp án =", best)
    return best


trace_max_path(build_tree([-10, 9, 20, None, None, 15, 7]))

vào -10
  vào 9
  [bước 1] node=9: L=0, R=0, through=9, best=9, return=9
  vào 20
    vào 15
    [bước 2] node=15: L=0, R=0, through=15, best=15, return=15
    vào 7
    [bước 3] node=7: L=0, R=0, through=7, best=15, return=7
  [bước 4] node=20: L=15, R=7, through=42, best=42, return=35
[bước 5] node=-10: L=9, R=35, through=34, best=42, return=25
Đáp án = 42


42

### 7.1 Trace thứ hai: thấy nhánh âm bị cắt

Cây `[1,-2,3,4,5,-6,2]`:

```text
            1
         /     \
       -2       3
      /  \     /  \
     4    5  -6    2
```

Dự đoán trước:

| Bước | Node | L | R | through | best sau | return |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | `4` | 0 | 0 | 4 | 4 | 4 |
| 2 | `5` | 0 | 0 | 5 | 5 | 5 |
| 3 | `-2` | 4 | 5 | 7 | 7 | 3 |
| 4 | `-6` | 0 | 0 | -6 | 7 | **-6** |
| 5 | `2` | 0 | 0 | 2 | 7 | 2 |
| 6 | `3` | **0** (cắt -6) | 2 | 5 | 7 | 5 |
| 7 | `1` | 3 | 5 | **9** | **9** | 6 |

- Bước 4: lá `-6` vẫn tự cập nhật `through = -6` (nó là đỉnh của đường `[-6]`) và trả `-6` lên — giá trị trả về **có thể âm**.
- Bước 6: cha `3` nhận `-6` rồi `max(-6, 0) = 0` → không đi xuống `-6`.
- Bước 3: node âm `-2` vẫn có ích: đường `4 -> -2 -> 5 = 7`. Ở bước 7 nó trả `3` (đoạn `-2 -> 5`) để root nối thành `5 -> -2 -> 1 -> 3 -> 2 = 9`.

In [22]:
trace_max_path(build_tree([1, -2, 3, 4, 5, -6, 2]))

vào 1
  vào -2
    vào 4
    [bước 1] node=4: L=0, R=0, through=4, best=4, return=4
    vào 5
    [bước 2] node=5: L=0, R=0, through=5, best=5, return=5
  [bước 3] node=-2: L=4, R=5, through=7, best=7, return=3
  vào 3
    vào -6
    [bước 4] node=-6: L=0, R=0, through=-6, best=7, return=-6
    vào 2
    [bước 5] node=2: L=0, R=0, through=2, best=7, return=2
  [bước 6] node=3: L=0, R=2, through=5, best=7, return=5
[bước 7] node=1: L=3, R=5, through=9, best=9, return=6
Đáp án = 9


9

## 8. Code nộp LeetCode

Chuyển `max_path_sum_dfs` vào đúng khung `class Solution`. Trên LeetCode **không** cần tự định nghĩa `TreeNode` (đã có sẵn); trong notebook ta đã định nghĩa ở mục 2.

Hai cách giữ biến `best` đều phổ biến:

- hàm lồng + `nonlocal best` (như dưới đây);
- hoặc `self.best`, nhưng nhớ **gán lại `self.best` ở đầu mỗi lần gọi `maxPathSum`** để không dính kết quả của test trước nếu cùng một object được dùng nhiều lần.

Ghi chú về độ sâu đệ quy: cây lệch có thể sâu tới `3 · 10^4` tầng, lớn hơn giới hạn đệ quy mặc định của Python (thường là 1000). Lời giải đệ quy này là lời giải phổ biến cho bài 124; nếu lo cây quá sâu, xem **cách 4** (bản lặp) ở mục 10.

In [23]:
class Solution:
    def maxPathSum(self, root: Optional[TreeNode]) -> int:
        best = float("-inf")

        def gain(node: Optional[TreeNode]) -> int:
            nonlocal best
            if node is None:
                return 0
            left_gain = max(gain(node.left), 0)
            right_gain = max(gain(node.right), 0)
            best = max(best, node.val + left_gain + right_gain)
            return node.val + max(left_gain, right_gain)

        gain(root)
        return best

In [24]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS ({actual})")


sol = Solution()
check("Example 1", sol.maxPathSum(build_tree([1, 2, 3])), 6)
check("Example 2", sol.maxPathSum(build_tree([-10, 9, 20, None, None, 15, 7])), 42)
check("một node âm", sol.maxPathSum(build_tree([-3])), -3)
check("một node dương", sol.maxPathSum(build_tree([1000])), 1000)
check("toàn âm", sol.maxPathSum(build_tree([-2, -1])), -1)
check("cắt nhánh âm", sol.maxPathSum(build_tree([2, -1])), 2)
check("cây lệch phải", sol.maxPathSum(build_tree([1, None, 2, None, 3])), 6)
check("đỉnh sâu trong cây",
      sol.maxPathSum(build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])), 48)
check("nhiều nhánh âm",
      sol.maxPathSum(build_tree([9, 6, -3, None, None, -6, 2, None, None, 2, None, -6, -6, -6])), 16)
check("gọi lại cùng object", sol.maxPathSum(build_tree([-3])), -3)

Example 1                    PASS (6)
Example 2                    PASS (42)
một node âm                  PASS (-3)
một node dương               PASS (1000)
toàn âm                      PASS (-1)
cắt nhánh âm                 PASS (2)
cây lệch phải                PASS (6)
đỉnh sâu trong cây           PASS (48)
nhiều nhánh âm               PASS (16)
gọi lại cùng object          PASS (-3)


Giải thích hai ca khó hơn:

- `[5,4,8,11,null,13,4,7,2,null,null,null,1]`: đường tốt nhất `7 -> 11 -> 4 -> 5 -> 8 -> 13 = 48`, đỉnh là root `5`.
- `[9,6,-3,null,null,-6,2,null,null,2,null,-6,-6,-6]`: đường tốt nhất `6 -> 9 -> -3 -> 2 -> 2 = 16` — phải đi qua node âm `-3` để nối hai phần dương, và bỏ hẳn các nhánh `-6`.

Ta in hai cây đó ra để tự kiểm tra bằng mắt:

In [25]:
print_tree(build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1]))
print()
print_tree(build_tree([9, 6, -3, None, None, -6, 2, None, None, 2, None, -6, -6, -6]))

5
├── L: 4
│   ├── L: 11
│   │   ├── L: 7
│   │   └── R: 2
│   └── R: ·
└── R: 8
    ├── L: 13
    └── R: 4
        ├── L: ·
        └── R: 1

9
├── L: 6
└── R: -3
    ├── L: -6
    └── R: 2
        ├── L: 2
        │   ├── L: -6
        │   │   ├── L: -6
        │   │   └── R: ·
        │   └── R: -6
        └── R: ·


## 9. Đối chiếu chéo các cách trên cây ngẫu nhiên

Ví dụ tự nghĩ thì dễ bỏ sót trường hợp. Ta sinh nhiều cây ngẫu nhiên nhỏ (giá trị từ `-10` đến `10`, có cả số âm), rồi so **cả ba cách** với nhau. Brute force (cách 1) chậm nhưng gần như "đúng theo định nghĩa", nên là trọng tài tốt.

`random.seed(124)` cố định chuỗi ngẫu nhiên → chạy lại notebook luôn ra đúng các cây đó.

Hàm `random_tree(n)`: lần lượt gắn từng node mới vào một chỗ trống (trái/phải) của một node cũ chọn ngẫu nhiên → ra đủ kiểu hình dạng.

In [26]:
def random_tree(n, lo=-10, hi=10):
    nodes = [TreeNode(random.randint(lo, hi))]
    while len(nodes) < n:
        parent = random.choice(nodes)
        child = TreeNode(random.randint(lo, hi))
        if random.random() < 0.5:
            if parent.left is None:
                parent.left = child
                nodes.append(child)
        else:
            if parent.right is None:
                parent.right = child
                nodes.append(child)
    return nodes[0]


random.seed(124)
for trial in range(500):
    t = random_tree(random.randint(1, 12))
    a = max_path_brute_graph(t)
    b = max_path_apex_naive(t)
    c = max_path_sum_dfs(t)
    d = Solution().maxPathSum(t)
    assert a == b == c == d, (tree_to_list(t), a, b, c, d)
print("500 cây ngẫu nhiên: brute graph == apex naive == DFS == Solution  ✔")

random.seed(7)
t = random_tree(8)
print("Một cây mẫu:", tree_to_list(t), "-> đáp án", max_path_sum_dfs(t))
print_tree(t)

500 cây ngẫu nhiên: brute graph == apex naive == DFS == Solution  ✔
Một cây mẫu: [0, 7, 2, None, -9, -9, None, -8, None, 3, -7] -> đáp án 9
0
├── L: 7
│   ├── L: ·
│   └── R: -9
│       ├── L: -8
│       └── R: ·
└── R: 2
    ├── L: -9
    │   ├── L: 3
    │   └── R: -7
    └── R: ·


## 10. Cách 4 (tùy chọn) — Bản lặp dùng stack, không lo giới hạn đệ quy

### 10.1 Vì sao cần?

Python giới hạn độ sâu đệ quy (mặc định thường là 1000) để tránh tràn bộ nhớ. Đề cho tới `3 · 10^4` node, nên một cây lệch hoàn toàn sẽ sâu `3 · 10^4` tầng → lời giải đệ quy chạy **cục bộ** có thể gặp `RecursionError`.

Hai cách gỡ:

1. `sys.setrecursionlimit(...)` tăng giới hạn — đơn giản nhưng phụ thuộc môi trường (stack của hệ điều hành vẫn có hạn).
2. Viết lại **không đệ quy**, tự quản lý thứ tự bằng **stack**. Đây là cách 4.

### 10.2 Stack là gì?

**Stack** (ngăn xếp) ngược với queue: **vào sau, ra trước** (LIFO — Last In, First Out), như chồng đĩa. Python dùng luôn `list`: `append` để đặt lên đỉnh, `pop()` để lấy ở đỉnh.

### 10.3 Mẹo lấy thứ tự "con trước cha"

1. Dùng stack duyệt cây theo kiểu "lấy node ra → ghi vào `order` → đẩy các con vào stack". Khi đó **mỗi node luôn được ghi vào `order` trước các con của nó** (vì con chỉ được đẩy vào stack sau khi cha đã được lấy ra).
2. Đảo ngược `order` → **mỗi con đứng trước cha** — đúng điều post-order cần.
3. Duyệt `reversed(order)`, tính `gain` của từng node từ `gain` của hai con (đã có sẵn trong một `dict`).

In [27]:
stack = []
stack.append("đĩa 1")
stack.append("đĩa 2")
stack.append("đĩa 3")
print("lấy ra:", stack.pop())     # đĩa 3: vào sau, ra trước
print("còn   :", stack)

# Thứ tự "cha trước con" bằng stack, rồi đảo ngược
order, st = [], [root2]
while st:
    node = st.pop()
    order.append(node)
    if node.left:
        st.append(node.left)
    if node.right:
        st.append(node.right)
print("cha trước con :", [x.val for x in order])
print("đảo -> con trước cha:", [x.val for x in reversed(order)])

lấy ra: đĩa 3
còn   : ['đĩa 1', 'đĩa 2']
cha trước con : [-10, 20, 7, 15, 9]
đảo -> con trước cha: [9, 15, 7, 20, -10]


In [28]:
def max_path_sum_iterative(root):
    order, stack = [], [root]
    while stack:                              # 1) thứ tự cha trước con
        node = stack.pop()
        order.append(node)
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)

    gain = {}                                 # node -> gain(node) (giá trị trả về)
    best = float("-inf")
    for node in reversed(order):              # 2) con luôn được xử lý trước cha
        left_gain = max(gain.get(node.left, 0), 0)     # None không có trong dict -> 0
        right_gain = max(gain.get(node.right, 0), 0)
        best = max(best, node.val + left_gain + right_gain)
        gain[node] = node.val + max(left_gain, right_gain)
    return best


print(max_path_sum_iterative(build_tree([1, 2, 3])))                         # 6
print(max_path_sum_iterative(build_tree([-10, 9, 20, None, None, 15, 7])))   # 42
print(max_path_sum_iterative(build_tree([-3])))                              # -3

random.seed(2024)
for _ in range(300):
    t = random_tree(random.randint(1, 30))
    assert max_path_sum_iterative(t) == max_path_sum_dfs(t)
print("bản lặp khớp bản đệ quy trên 300 cây ngẫu nhiên")

6
42
-3
bản lặp khớp bản đệ quy trên 300 cây ngẫu nhiên


### 10.4 Thử với cây lệch `3 · 10^4` node (đúng giới hạn đề)

Cây lệch toàn giá trị `1` thì đáp án là cả đường thẳng: `30000`. Bản đệ quy (chạy với giới hạn đệ quy mặc định) sẽ báo `RecursionError`; bản lặp chạy bình thường.

In [29]:
deep = make_chain(30000, val=1)
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())
try:
    max_path_sum_dfs(deep)
except RecursionError as e:
    print("Bản đệ quy: RecursionError -", str(e)[:45], "...")
print("Bản lặp   :", max_path_sum_iterative(deep))

giới hạn đệ quy hiện tại: 1000
Bản đệ quy: RecursionError - maximum recursion depth exceeded ...
Bản lặp   : 30000


In [30]:
# 10.5 Bản lặp theo format LeetCode (tùy chọn)
class SolutionIterative:
    def maxPathSum(self, root: Optional[TreeNode]) -> int:
        order, stack = [], [root]
        while stack:
            node = stack.pop()
            order.append(node)
            if node.left:
                stack.append(node.left)
            if node.right:
                stack.append(node.right)
        gain = {}
        best = float("-inf")
        for node in reversed(order):
            left_gain = max(gain.get(node.left, 0), 0)
            right_gain = max(gain.get(node.right, 0), 0)
            best = max(best, node.val + left_gain + right_gain)
            gain[node] = node.val + max(left_gain, right_gain)
        return best


assert SolutionIterative().maxPathSum(build_tree([1, 2, 3])) == 6
assert SolutionIterative().maxPathSum(build_tree([-10, 9, 20, None, None, 15, 7])) == 42
assert SolutionIterative().maxPathSum(build_tree([-2, -1])) == -1
assert SolutionIterative().maxPathSum(make_chain(30000, val=-1)) == -1
print("SolutionIterative PASS")

SolutionIterative PASS


## 11. Vì sao cách 3 đúng?

**Mệnh đề 1 (giá trị trả về đúng).** Với mọi node `x`, `gain(x)` trả về đúng "tổng lớn nhất của đường đi bắt đầu tại `x` và chỉ đi xuống".

Chứng minh theo kiểu "từ lá lên" (quy nạp theo chiều cao):

- Lá: chỉ có một đường đi xuống là `[x]`; hàm trả `x.val + max(0, 0) = x.val`. Đúng.
- Node `x` bất kỳ: giả sử hai con đã trả đúng. Một đường đi xuống từ `x` hoặc dừng ở `x` (tổng `x.val`), hoặc đi tiếp vào đúng **một** con và từ đó là một đường đi xuống của con → tốt nhất là `x.val + gain(con)`. Lấy max của ba khả năng chính là `x.val + max(L, R)` với `L, R` đã kẹp ở 0. Đúng.

**Mệnh đề 2 (không bỏ sót).** Mỗi đường đi `P` có đúng một đỉnh `x` (mục 5.1). `P` = (đoạn xuống trái) + `x` + (đoạn xuống phải), nên `sum(P) <= x.val + L + R = through(x)`. Khi DFS thăm `x`, `best` được so với `through(x)`, nên `best >= sum(P)`.

**Mệnh đề 3 (không trả số "ảo").** Mỗi giá trị `through(x)` đều là tổng của một đường đi **thật**: đoạn trái nằm trong cây con trái, đoạn phải nằm trong cây con phải — hai tập rời nhau, cộng thêm `x` → không node nào lặp, không rẽ nhánh. Nên `best` không bao giờ vượt quá đáp án thật.

Từ 2 và 3: `best` **đúng bằng** tổng đường đi lớn nhất.

**Invariant bằng lời**: *"Sau khi DFS xong cây con gốc `x`: hàm đã trả về gain một nhánh tốt nhất của `x`, và `best` đã là đáp án tốt nhất của mọi đường đi có đỉnh nằm trong cây con đó."*

## 12. Độ phức tạp và so sánh các cách

`n` = số node, `h` = chiều cao cây (`log n <= h <= n`).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force đồ thị | DFS từ mọi node, mỗi node chạm tới là một đầu mút | `O(n²)` | `O(n)` | dễ nghĩ, làm trọng tài để test; TLE với `n = 3·10^4` |
| 2. Thử từng đỉnh | `through(x)` với `max_gain` tính lại | `O(n²)` worst (cây lệch), `O(n log n)` cây cân bằng | `O(h)` | đúng ý tưởng nhưng lặp công việc |
| 3. **DFS post-order một lượt** | trả gain 1 nhánh, cập nhật best 2 nhánh | **`O(n)`** | `O(h)` call stack | **cách chính để nộp** |
| 4. Lặp bằng stack | thứ tự con-trước-cha + dict gain | `O(n)` | `O(n)` | không bị giới hạn đệ quy |

Vì sao cách 3 là `O(n)`: mỗi node được gọi `gain` đúng một lần, mỗi lần làm `O(1)` việc (vài phép cộng và `max`); cộng thêm `n + 1` lời gọi với `None` cũng chỉ là `O(n)`. Bộ nhớ là độ sâu call stack, tối đa bằng chiều cao `h` (cây lệch thì `h = n`).

Đo thời gian thực tế trên một cây ngẫu nhiên 1500 node (con số cụ thể tùy máy, chỉ cần nhìn **độ chênh**):

In [31]:
random.seed(1)
big = random_tree(1500, -1000, 1000)
print("chiều cao cây:", height(big))

for name, fn in [("1. brute graph  ", max_path_brute_graph),
                 ("2. apex naive   ", max_path_apex_naive),
                 ("3. DFS post-order", max_path_sum_dfs),
                 ("4. iterative    ", max_path_sum_iterative)]:
    t0 = time.perf_counter()
    ans = fn(big)
    print(f"{name}: đáp án = {ans}, thời gian ≈ {(time.perf_counter() - t0) * 1000:8.2f} ms")

chiều cao cây: 26


1. brute graph  : đáp án = 12595, thời gian ≈   340.01 ms
2. apex naive   : đáp án = 12595, thời gian ≈     2.56 ms
3. DFS post-order: đáp án = 12595, thời gian ≈     0.38 ms
4. iterative    : đáp án = 12595, thời gian ≈     0.61 ms


## 13. Lỗi thường gặp

### Lỗi 1: Trả về cả hai nhánh cho cha

```python
return node.val + left_gain + right_gain     # SAI
```

Cha sẽ nối thêm vào một đoạn đã rẽ đôi → tạo hình chữ Y. Ví dụ cây `[1,2,null,3,4]`:

```text
      1
     /
    2
   / \
  3   4
```

Đáp án đúng là `3 -> 2 -> 4 = 9`. Bản sai: node `2` trả `9` lên, root tính `1 + 9 = 10` — tương ứng "đường" `{3, 2, 4, 1}` mà node `2` nối 3 node. Sửa: `return node.val + max(left_gain, right_gain)`.

In [32]:
def max_path_wrong_two_branches(root):
    best = float("-inf")

    def gain(node):
        nonlocal best
        if node is None:
            return 0
        l = max(gain(node.left), 0)
        r = max(gain(node.right), 0)
        best = max(best, node.val + l + r)
        return node.val + l + r                  # SAI: trả 2 nhánh

    gain(root)
    return best


t = build_tree([1, 2, None, 3, 4])
print("sai :", max_path_wrong_two_branches(t))   # 10
print("đúng:", max_path_sum_dfs(t))              # 9

sai : 10
đúng: 9


### Lỗi 2: Khởi tạo `best = 0`

Cây toàn âm sẽ ra `0` — tương ứng đường đi rỗng, đề không cho phép. Sửa: `best = float("-inf")` hoặc `best = root.val`.

### Lỗi 3: Quên `max(..., 0)` cho gain của con

Không kẹp ở 0 thì nhánh âm vẫn bị cộng vào. Cây `[2,-1]`: con trái trả `-1`, root tính `2 + (-1) + 0 = 1` thay vì `2`.

### Lỗi 4: Kẹp cả `node.val` khi cập nhật

```python
best = max(best, max(node.val + l + r, 0))   # SAI
```

Lại ngầm cho phép "đường rỗng" có tổng 0: cây `[-3]` ra `0`. Chỉ kẹp **gain của con**, không kẹp ứng viên đáp án.

Cell dưới chạy cả ba phiên bản sai trên đúng các ca làm lộ lỗi:

In [33]:
def make_variant(init_best, clip_child, clip_update):
    def solve(root):
        best = init_best

        def gain(node):
            nonlocal best
            if node is None:
                return 0
            l, r = gain(node.left), gain(node.right)
            if clip_child:
                l, r = max(l, 0), max(r, 0)
            cand = node.val + l + r
            best = max(best, max(cand, 0) if clip_update else cand)
            return node.val + max(l, r)

        gain(root)
        return best
    return solve


wrong_init_zero = make_variant(0, True, False)                 # Lỗi 2
wrong_no_clip = make_variant(float("-inf"), False, False)      # Lỗi 3
wrong_clip_update = make_variant(float("-inf"), True, True)    # Lỗi 4

print("Lỗi 2, cây [-3]  :", wrong_init_zero(build_tree([-3])), " | đúng:", max_path_sum_dfs(build_tree([-3])))
print("Lỗi 3, cây [2,-1]:", wrong_no_clip(build_tree([2, -1])), " | đúng:", max_path_sum_dfs(build_tree([2, -1])))
print("Lỗi 4, cây [-3]  :", wrong_clip_update(build_tree([-3])), " | đúng:", max_path_sum_dfs(build_tree([-3])))

Lỗi 2, cây [-3]  : 0  | đúng: -3
Lỗi 3, cây [2,-1]: 1  | đúng: 2
Lỗi 4, cây [-3]  : 0  | đúng: -3


### Lỗi 5: Trả về giá trị `gain(root)` thay vì `best`

```python
return gain(root)     # SAI
```

`gain(root)` chỉ là đường đi **bắt đầu tại root và đi xuống**. Example 2 sẽ ra `25` thay vì `42`. Hàm đệ quy được gọi để **cập nhật `best`**; giá trị nó trả ở root bị bỏ đi.

### Lỗi 6: Quên `nonlocal best`

Gán `best = ...` trong hàm lồng mà không có `nonlocal` → `UnboundLocalError` (đã thấy ở mục 3.6).

### Lỗi 7: Dùng `self.best` nhưng không reset

Nếu viết `best = float("-inf")` ở mức **class** (thuộc tính class) rồi cập nhật `self.best`, thì khi cùng một object `Solution` được gọi cho nhiều cây, kết quả cây trước còn dính lại. Luôn gán `self.best = float("-inf")` **ở đầu** `maxPathSum`.

### Lỗi 8: Đi tìm "max root-to-leaf" hoặc bắt buộc đi qua root

Đường đi có thể bắt đầu, kết thúc ở bất kỳ đâu và không cần qua root. `best_root_to_leaf` ở mục 3.5 cho Example 2 ra `25`, sai.

In [34]:
class SolutionSelfBestBuggy:
    best = float("-inf")                  # thuộc tính class, không reset

    def maxPathSum(self, root):
        def gain(node):
            if node is None:
                return 0
            l = max(gain(node.left), 0)
            r = max(gain(node.right), 0)
            self.best = max(self.best, node.val + l + r)
            return node.val + max(l, r)
        gain(root)
        return self.best


s = SolutionSelfBestBuggy()
print("cây Example 2:", s.maxPathSum(build_tree([-10, 9, 20, None, None, 15, 7])))   # 42
print("cây [-3]     :", s.maxPathSum(build_tree([-3])), "<- SAI, đúng phải là -3")   # 42 dính lại

cây Example 2: 42
cây [-3]     : 42 <- SAI, đúng phải là -3


## 14. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu "tree DP: trả về một thứ, cập nhật toàn cục thứ khác"

- Đề hỏi về **đường đi trong cây** mà **không bắt buộc qua root** ("any path", "between any two nodes", "longest path").
- Đường đi đó có thể **gập** tại một node (đi lên rồi đi xuống).
- Đáp án tại một node cần thông tin của **cả hai con** → post-order.

Khi thấy các dấu hiệu này, nghĩ ngay tới khung:

```text
def dfs(node):
    if node is None: return <giá trị trung tính>
    a = dfs(node.left); b = dfs(node.right)
    best = max(best, <kết hợp CẢ HAI nhánh tại node>)     # đường gập tại node
    return <kéo dài MỘT nhánh lên cha>                    # đường còn nối tiếp được
```

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 543. Diameter of Binary Tree | y hệt khung, nhưng đếm **số cạnh** thay vì cộng giá trị (mọi "giá trị" ≥ 0 nên không cần kẹp) |
| 687. Longest Univalue Path | chỉ nối nhánh con khi con có **cùng giá trị** với node |
| 2246. Longest Path With Different Adjacent Characters | cây tổng quát nhiều con: chọn **2 nhánh lớn nhất** trong các con |
| 1372. Longest ZigZag Path in a Binary Tree | trả về hai số (đi trái/đi phải) thay vì một |
| 112 / 113 / 437. Path Sum I / II / III | đường đi từ trên xuống (bài 437 dùng prefix sum) |
| 337. House Robber III | post-order trả về cặp giá trị (lấy / không lấy node) |
| 968. Binary Tree Cameras | post-order trả về "trạng thái" của node cho cha quyết định |

Ví dụ chuyển giao ngay: **543. Diameter of Binary Tree** — độ dài (số cạnh) đường đi dài nhất giữa hai node bất kỳ. Chỉ thay "giá trị" bằng "độ dài":

In [35]:
def diameter_of_binary_tree(root):
    best = 0

    def depth(node):                       # số node trên nhánh xuống dài nhất
        nonlocal best
        if node is None:
            return 0
        l = depth(node.left)
        r = depth(node.right)
        best = max(best, l + r)            # gập tại node: l + r cạnh
        return 1 + max(l, r)               # kéo dài một nhánh lên cha

    depth(root)
    return best


print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])))   # 3: 4 -> 2 -> 1 -> 3
print(diameter_of_binary_tree(build_tree([1, 2])))            # 1

3
1


## 15. Bài tập vận dụng

Làm theo thứ tự, mỗi bài khó hơn bài trước. Gợi ý ở mục 16, đáp án ở mục 17 (cuối notebook) — hãy tự làm trước!

### Bài 1 — Đoán output (không chạy code)

```python
def f(node):
    if node is None:
        return 0
    a = f(node.left)
    b = f(node.right)
    print(node.val, a, b)
    return node.val + max(a, b)

root = build_tree([1, 2, 3, None, 4])
print("kết quả:", f(root))
```

Viết ra 5 dòng output theo đúng thứ tự và giải thích vì sao node `1` được in **cuối cùng**.

### Bài 2 — Tính tay bằng bảng

Cho cây `[2,-1,3,null,4,-5]`. Hãy vẽ cây, rồi lập bảng giống mục 7 (Bước, Node, L, R, through, best sau, return) theo thứ tự post-order. Đáp án là bao nhiêu, ứng với đường đi nào? Node nào bị "cắt"?

### Bài 3 — Điền TODO: lời giải chính

Hoàn thiện hàm theo cách 3 (một lượt DFS post-order).

In [36]:
def max_path_sum_exercise(root):
    best = float("-inf")

    def gain(node):
        # TODO 1: khai báo để gán lại được biến best ở hàm ngoài
        # TODO 2: base case: node rỗng trả về bao nhiêu?
        # TODO 3: left_gain, right_gain = gain của hai con, kẹp ở 0
        # TODO 4: cập nhật best bằng đường gập tại node (2 nhánh)
        # TODO 5: trả về đường một nhánh cho cha
        pass

    gain(root)
    return best


# Tự làm xong rồi mới mở test:
# assert max_path_sum_exercise(build_tree([1, 2, 3])) == 6
# assert max_path_sum_exercise(build_tree([-10, 9, 20, None, None, 15, 7])) == 42
# assert max_path_sum_exercise(build_tree([-3])) == -3
# assert max_path_sum_exercise(build_tree([2, -1])) == 2

### Bài 4 — Biến thể: đường đi **bắt buộc chứa root**

Viết `max_path_through_root(root)`: tổng lớn nhất của một đường đi (khác rỗng) **có chứa node root**.

Gợi ý nhỏ: root có thể là một đầu mút hoặc nằm giữa đường. Đây chính là `through(root)` ở mục 5.3.

```python
max_path_through_root([-10,9,20,null,null,15,7]) -> 34   # 9 -> -10 -> 20 -> 15
max_path_through_root([1,2,3]) -> 6
max_path_through_root([-3]) -> -3
max_path_through_root([2,-1]) -> 2
```

In [37]:
def max_path_through_root(root):
    # TODO: tính gain một nhánh cho hai con của root, kẹp ở 0, rồi ghép với root.val
    raise NotImplementedError


# Gợi ý test:
# assert max_path_through_root(build_tree([-10, 9, 20, None, None, 15, 7])) == 34
# assert max_path_through_root(build_tree([1, 2, 3])) == 6
# assert max_path_through_root(build_tree([-3])) == -3
# assert max_path_through_root(build_tree([2, -1])) == 2

### Bài 5 — Biến thể: LeetCode 687. Longest Univalue Path

Cho `root`, trả về **độ dài** (số **cạnh**) của đường đi dài nhất mà **mọi node trên đường đi có cùng giá trị**. Đường đi không bắt buộc qua root. (Theo đề 687: số node trong `[0, 10^4]`, tức là cây có thể rỗng → trả `0`; `-1000 <= Node.val <= 1000`; độ sâu cây không quá 1000.)

```text
Input: root = [5,4,5,1,1,null,5]   Output: 2   (5 -> 5 -> 5 ở bên phải)
Input: root = [1,4,5,4,4,null,5]   Output: 2   (4 -> 4 -> 4 ở bên trái)
```

Khung y hệt bài 124: hàm đệ quy trả về **độ dài nhánh xuống dài nhất có cùng giá trị với node**, và cập nhật `best` bằng **tổng hai nhánh**. Điểm mới: một nhánh con chỉ được nối nếu con **cùng giá trị** với node.

In [38]:
def longest_univalue_path(root):
    # TODO: best = 0; dfs(node) trả về số cạnh của nhánh xuống dài nhất cùng giá trị node
    pass


# Gợi ý test:
# assert longest_univalue_path(build_tree([5, 4, 5, 1, 1, None, 5])) == 2
# assert longest_univalue_path(build_tree([1, 4, 5, 4, 4, None, 5])) == 2
# assert longest_univalue_path(build_tree([])) == 0
# assert longest_univalue_path(build_tree([1, 1, 1, 1])) == 3

### Bài 6 — Mini challenge: trả về **chính đường đi** có tổng lớn nhất

Viết `max_path_nodes(root)` trả về **list giá trị** của một đường đi có tổng lớn nhất, theo đúng thứ tự dọc đường đi (từ đầu mút này sang đầu mút kia). Yêu cầu vẫn `O(n)` số lần thăm node (không gọi lại `max_gain` nhiều lần) — chấp nhận việc nối list tốn thêm chi phí.

```python
max_path_nodes([-10,9,20,null,null,15,7]) -> [15, 20, 7]   (hoặc [7, 20, 15])
max_path_nodes([1,2,3])                   -> [2, 1, 3]     (hoặc [3, 1, 2])
max_path_nodes([-3])                      -> [-3]
```

Gợi ý nhỏ: cho hàm đệ quy trả về **hai thứ**: gain và chính đoạn đi xuống (list giá trị, bắt đầu từ node).

In [39]:
def max_path_nodes(root):
    # TODO: dfs(node) -> (gain, chain) với chain = [node.val, ...] đi xuống theo nhánh tốt nhất
    # TODO: tại mỗi node, đường gập = đảo(chain trái) + [node.val] + chain phải (bỏ nhánh có gain <= 0)
    pass


# Gợi ý test:
# assert max_path_nodes(build_tree([-10, 9, 20, None, None, 15, 7])) in ([15, 20, 7], [7, 20, 15])
# assert max_path_nodes(build_tree([1, 2, 3])) in ([2, 1, 3], [3, 1, 2])
# assert max_path_nodes(build_tree([-3])) == [-3]

## 16. Gợi ý theo mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | `print` nằm **sau** hai lời gọi đệ quy → post-order. | Vẽ cây: `1` có con trái `2`, con phải `3`; `2` có con phải `4`. | Thứ tự in: `4`, `2`, `3`, `1`; `None` trả 0. |
| 2 | Vẽ cây trước, rồi đi post-order. | Post-order: `4, -1, -5, 3, 2`. | Ở `3`: con trái `-5` trả `-5` → `max(-5, 0) = 0`. |
| 3 | Đúng bảng "trả về / cập nhật" ở mục 6.2. | `nonlocal best`; `if node is None: return 0`. | `best = max(best, node.val + l + r)`; `return node.val + max(l, r)`. |
| 4 | Root là đỉnh của đường đi. | Cần một hàm gain như `max_gain` ở mục 5.2 (gọi 2 lần là đủ, vẫn `O(n)`). | `root.val + max(0, gain(root.left)) + max(0, gain(root.right))`. |
| 5 | "Giá trị trả về" = độ dài nhánh cùng màu; "cập nhật" = tổng hai nhánh. | Luôn gọi `dfs` cho **cả hai con** (kể cả khi khác giá trị), vì đường tốt nhất có thể nằm hẳn trong cây con. | `l = dfs(left) + 1 if left and left.val == node.val else 0` (gọi `dfs(left)` trước). |
| 6 | Mỗi lần cập nhật `best`, lưu luôn **đường đi** tương ứng. | `dfs` trả `(gain, chain)`; nhánh có gain `<= 0` thì chain coi như `[]`. | `path = left_chain[::-1] + [node.val] + right_chain`; `return (node.val + g, [node.val] + chain_tốt_hơn)`. |

## 17. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
4 0 0
2 0 4
3 0 0
1 6 3
kết quả: 7
```

Cây:

```text
      1
     / \
    2   3
     \
      4
```

- `f(4)`: hai con `None` trả 0 → in `4 0 0`, trả `4`.
- `f(2)`: con trái `None` → `a = 0`; con phải là `4` → `b = 4`. In `2 0 4`, trả `2 + 4 = 6`.
- `f(3)`: in `3 0 0`, trả `3`.
- `f(1)`: `a = 6`, `b = 3` → in `1 6 3`, trả `1 + 6 = 7`.

Node `1` in cuối cùng vì dòng `print` nằm **sau** hai lời gọi đệ quy (post-order): root phải chờ cả hai cây con xong. (Hàm `f` không kẹp ở 0, nên nó tính "đường xuống tới lá lớn nhất" khi mọi giá trị dương.)

### Đáp án Bài 2

Cây `[2,-1,3,null,4,-5]`:

```text
        2
      /   \
    -1     3
      \   /
       4 -5
```

| Bước | Node | L | R | through | best sau | return |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | `4` | 0 | 0 | 4 | 4 | 4 |
| 2 | `-1` | 0 | 4 | 3 | 4 | 3 |
| 3 | `-5` | 0 | 0 | -5 | 4 | -5 |
| 4 | `3` | **0** (cắt -5) | 0 | 3 | 4 | 3 |
| 5 | `2` | 3 | 3 | **8** | **8** | 5 |

Đáp án `8`, đường đi `4 -> -1 -> 2 -> 3`. Nhánh `-5` bị cắt ở bước 4. Node âm `-1` vẫn được dùng vì nó là "cầu nối" để lấy được `4`.

Cell kế tiếp kiểm chứng Bài 2 bằng `trace_max_path`; cell sau nữa là **đáp án Bài 3–6** kèm assert.

In [40]:
print_tree(build_tree([2, -1, 3, None, 4, -5]))
trace_max_path(build_tree([2, -1, 3, None, 4, -5]))

2
├── L: -1
│   ├── L: ·
│   └── R: 4
└── R: 3
    ├── L: -5
    └── R: ·
vào 2
  vào -1
    vào 4
    [bước 1] node=4: L=0, R=0, through=4, best=4, return=4
  [bước 2] node=-1: L=0, R=4, through=3, best=4, return=3
  vào 3
    vào -5
    [bước 3] node=-5: L=0, R=0, through=-5, best=4, return=-5
  [bước 4] node=3: L=0, R=0, through=3, best=4, return=3
[bước 5] node=2: L=3, R=3, through=8, best=8, return=5
Đáp án = 8


8

In [41]:
def answer_max_path_sum_exercise(root):
    best = float("-inf")

    def gain(node):
        nonlocal best                                   # TODO 1
        if node is None:                                # TODO 2
            return 0
        left_gain = max(gain(node.left), 0)             # TODO 3
        right_gain = max(gain(node.right), 0)
        best = max(best, node.val + left_gain + right_gain)   # TODO 4
        return node.val + max(left_gain, right_gain)    # TODO 5

    gain(root)
    return best


def answer_max_path_through_root(root):
    def gain(node):
        if node is None:
            return 0
        return node.val + max(0, gain(node.left), gain(node.right))

    return root.val + max(0, gain(root.left)) + max(0, gain(root.right))


def answer_longest_univalue_path(root):
    best = 0

    def dfs(node):
        nonlocal best
        if node is None:
            return 0
        left_len = dfs(node.left)          # luôn đi xuống cả hai con
        right_len = dfs(node.right)
        l = left_len + 1 if node.left and node.left.val == node.val else 0
        r = right_len + 1 if node.right and node.right.val == node.val else 0
        best = max(best, l + r)            # gập tại node
        return max(l, r)                   # một nhánh cho cha

    dfs(root)
    return best


def answer_max_path_nodes(root):
    best_sum = float("-inf")
    best_path = []

    def dfs(node):
        nonlocal best_sum, best_path
        if node is None:
            return 0, []
        lg, lchain = dfs(node.left)
        rg, rchain = dfs(node.right)
        if lg <= 0:
            lg, lchain = 0, []
        if rg <= 0:
            rg, rchain = 0, []
        total = node.val + lg + rg
        if total > best_sum:
            best_sum = total
            best_path = lchain[::-1] + [node.val] + rchain
        if lg >= rg:
            return node.val + lg, [node.val] + lchain
        return node.val + rg, [node.val] + rchain

    dfs(root)
    return best_path


ex2_tree = build_tree([-10, 9, 20, None, None, 15, 7])

assert answer_max_path_sum_exercise(build_tree([1, 2, 3])) == 6
assert answer_max_path_sum_exercise(ex2_tree) == 42
assert answer_max_path_sum_exercise(build_tree([-3])) == -3
assert answer_max_path_sum_exercise(build_tree([2, -1])) == 2

assert answer_max_path_through_root(ex2_tree) == 34
assert answer_max_path_through_root(build_tree([1, 2, 3])) == 6
assert answer_max_path_through_root(build_tree([-3])) == -3
assert answer_max_path_through_root(build_tree([2, -1])) == 2

assert answer_longest_univalue_path(build_tree([5, 4, 5, 1, 1, None, 5])) == 2
assert answer_longest_univalue_path(build_tree([1, 4, 5, 4, 4, None, 5])) == 2
assert answer_longest_univalue_path(build_tree([])) == 0
assert answer_longest_univalue_path(build_tree([1, 1, 1, 1])) == 3

assert answer_max_path_nodes(ex2_tree) in ([15, 20, 7], [7, 20, 15])
assert answer_max_path_nodes(build_tree([1, 2, 3])) in ([2, 1, 3], [3, 1, 2])
assert answer_max_path_nodes(build_tree([-3])) == [-3]
print("Bài 6, Example 2:", answer_max_path_nodes(ex2_tree))


# Đối chiếu ngẫu nhiên: Bài 3 và Bài 6 với brute force.
# Bài 6 dùng giá trị KHÁC NHAU để kiểm tra đường đi trả về có thật sự là đường đi hợp lệ.
def is_valid_path(root, vals):
    graph = tree_to_graph(root)
    by_val = {node.val: node for node in graph}
    nodes = [by_val[v] for v in vals]
    if len(set(vals)) != len(vals):
        return False
    return all(nodes[i + 1] in graph[nodes[i]] for i in range(len(nodes) - 1))


random.seed(687)
for _ in range(300):
    n = random.randint(1, 12)
    t = random_tree(n)
    assert answer_max_path_sum_exercise(t) == max_path_brute_graph(t)
    distinct = random.sample(range(-30, 31), n)            # n giá trị đôi một khác nhau
    nodes = list(tree_to_graph(t).keys())
    for node, v in zip(nodes, distinct):
        node.val = v
    path = answer_max_path_nodes(t)
    assert sum(path) == max_path_brute_graph(t) and is_valid_path(t, path)

print("Đáp án tham khảo chạy PASS toàn bộ test!")

Bài 6, Example 2: [15, 20, 7]
Đáp án tham khảo chạy PASS toàn bộ test!


## 18. Template cuối cùng nên nhớ

**DFS post-order một lượt (cách chính)** — `O(n)` time, `O(h)` space:

```python
class Solution:
    def maxPathSum(self, root: Optional[TreeNode]) -> int:
        best = float("-inf")

        def gain(node):
            nonlocal best
            if node is None:
                return 0
            l = max(gain(node.left), 0)          # nhánh âm thì bỏ
            r = max(gain(node.right), 0)
            best = max(best, node.val + l + r)   # gập tại node: 2 nhánh -> cập nhật
            return node.val + max(l, r)          # nối lên cha: 1 nhánh -> trả về

        gain(root)
        return best
```

**Câu thần chú**: *"Hỏi con trước. Âm thì bỏ. Hai nhánh để cập nhật, một nhánh để trả về. Trả lời bằng `best`, không bằng giá trị return."*

**Khung chung cho họ bài "đường đi trong cây"**: `dfs` trả về *phần còn nối dài được* (một nhánh), còn `best` ghi nhận *phần đã khép lại* (gập tại node, hai nhánh).

## 19. Checklist tự kiểm tra trước khi submit

- `best` có khởi tạo là `float("-inf")` (hoặc `root.val`), **không phải** `0`, chưa?
- Gain của mỗi con có được kẹp `max(..., 0)` chưa? Có lỡ kẹp luôn ứng viên `node.val + l + r` không?
- Giá trị **trả về** chỉ dùng **một** nhánh (`max(l, r)`), giá trị **cập nhật** dùng **cả hai** (`l + r`) chưa?
- Có `nonlocal best` (hoặc `self.best` được reset ở đầu hàm) chưa?
- Hàm `maxPathSum` trả về `best`, không phải `gain(root)`, đúng không?
- Đã test: cây 1 node âm (`[-3]`), cây toàn âm (`[-2,-1]`), nhánh âm cần cắt (`[2,-1]`), đáp án không qua root (Example 2), cây lệch chưa?
- Nếu lo cây rất sâu (cây lệch `3·10^4` node), đã cân nhắc bản lặp dùng stack hoặc tăng giới hạn đệ quy chưa?